# 06 — Evaluate Al-Balad Qwen3-VL-2B

**Purpose:** Final held-out test-set evaluation for the Al-Balad historic-building image-description model.

This notebook follows `05_fine_tuning_albalad_qwen3vl2b_research_v2.ipynb` and compares:

1. **Base:** `Qwen/Qwen3-VL-2B-Instruct`
2. **Fine-tuned:** the same base model with the trained Al-Balad LoRA adapter enabled

Both models receive the **same frozen test images, prompts, processor, 4-bit backbone, and deterministic decoding settings**.

### Main metrics

- Corpus BLEU-4
- Mean sentence BLEU-4
- ROUGE-L F1
- METEOR
- BERTScore F1
- CIDEr *(single-reference; interpret cautiously)*
- Building-identification accuracy
- Macro building-identification accuracy
- Day/night performance
- Per-building performance
- Paired bootstrap 95% confidence intervals
- Wilcoxon paired tests
- Exact McNemar test for building-identification changes

### Research outputs

The notebook saves complete predictions, per-image metrics, paper-ready tables, 300-DPI figures, objective qualitative examples, a blinded human-evaluation sheet, and an automatically generated numerical results summary.

> **Interpretation limitation:** The reference captions in this project are synthetic descriptions generated by Qwen3-VL using building metadata and survey context. Automatic caption-similarity scores therefore measure agreement with those references; they are not independent proof of historical factual correctness. Historical factuality and visual grounding should be supported by human evaluation and/or authoritative metadata.

## 1. Install evaluation dependencies

In [ ]:
!pip install -q \
    transformers==5.14.1 \
    peft==0.19.1 \
    accelerate==1.14.0 \
    bitsandbytes==0.49.2 \
    bert-score==0.3.13 \
    rouge-score==0.1.2 \
    nltk==3.9.2 \
    pycocoevalcap==1.2

In [ ]:
!pip install bert-score==0.3.13

In [ ]:
!pip install rouge_score

In [ ]:
!pip install bitsandbytes

## 2. Imports, Drive, GPU, reproducibility, and NLTK resources

In [ ]:
import os, gc, re, json, math, random, hashlib, shutil, platform, subprocess, warnings
from pathlib import Path
from contextlib import nullcontext
from collections import defaultdict
from textwrap import fill

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import scipy
import sklearn
import nltk
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import display

import transformers, peft, accelerate, bitsandbytes
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import PeftModel

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
from scipy.stats import binomtest, wilcoxon
from google.colab import drive

drive.mount('/content/drive')

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Choose Runtime > Change runtime type > GPU.')

subprocess.run(['nvidia-smi'], check=False)

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

versions = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'peft': peft.__version__,
    'accelerate': accelerate.__version__,
    'bitsandbytes': bitsandbytes.__version__,
    'nltk': nltk.__version__,
    'scipy': scipy.__version__,
    'scikit-learn': sklearn.__version__,
    'pandas': pd.__version__,
    'numpy': np.__version__,
}
display(pd.DataFrame(versions.items(), columns=['package','version']))

## 3. Evaluation configuration

In [ ]:
EXPERIMENT_NAME = 'qwen3_vl_2b_albalad_v1'
RUN_ROOT = Path('/content/drive/MyDrive/albalad_runs_v1') / EXPERIMENT_NAME

FINAL_ADAPTER_DIR = RUN_ROOT / 'final_lora_adapter'
PROCESSOR_DIR = RUN_ROOT / 'processor'
SPLITS_DIR = RUN_ROOT / 'splits'
TEST_JSONL = SPLITS_DIR / 'test.jsonl'
SPLIT_MANIFEST_PATH = SPLITS_DIR / 'split_manifest.csv'
SPLIT_METADATA_PATH = SPLITS_DIR / 'split_metadata.json'
EXPERIMENT_CONFIG_PATH = RUN_ROOT / 'experiment_config.json'
RUN_STATUS_PATH = RUN_ROOT / 'run_status.json'

EVALUATION_ROOT = RUN_ROOT / 'evaluation_v1'
PREDICTIONS_DIR = EVALUATION_ROOT / 'predictions'
METRICS_DIR = EVALUATION_ROOT / 'metrics'
FIGURES_DIR = EVALUATION_ROOT / 'figures'
QUALITATIVE_DIR = EVALUATION_ROOT / 'qualitative_examples'
HUMAN_EVAL_DIR = EVALUATION_ROOT / 'human_evaluation'

IMAGE_ROOT = Path(
    "/content/drive/MyDrive/albalad_image_caption/processed_dataset"
)

for d in [EVALUATION_ROOT, PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR, QUALITATIVE_DIR, HUMAN_EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Deterministic generation for fair paired comparison.
GENERATION_BATCH_SIZE = 2
MAX_NEW_TOKENS = 220
GENERATION_KWARGS = {
    'max_new_tokens': MAX_NEW_TOKENS,
    'do_sample': False,
    'num_beams': 1,
    'use_cache': True,
    'repetition_penalty': 1.05,
}

CACHE_TEST_IMAGES_LOCALLY = True
LOCAL_IMAGE_CACHE = Path('/content/albalad_evaluation_images')

BERTSCORE_MODEL = 'roberta-large'
BERTSCORE_BATCH_SIZE = 16
BERTSCORE_RESCALE_WITH_BASELINE = True

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_CONFIDENCE = 0.95
HUMAN_EVAL_SAMPLE_SIZE = 60

print('Run root:', RUN_ROOT)
print('Test JSONL:', TEST_JSONL)
print('Evaluation output:', EVALUATION_ROOT)

## 4. Validate the completed training experiment and frozen test split

In [ ]:
required = {
    'Experiment configuration': EXPERIMENT_CONFIG_PATH,
    'Run status': RUN_STATUS_PATH,
    'Final LoRA adapter': FINAL_ADAPTER_DIR,
    'Processor': PROCESSOR_DIR,
    'Frozen test JSONL': TEST_JSONL,
    'Split manifest': SPLIT_MANIFEST_PATH,
    'Split metadata': SPLIT_METADATA_PATH,
}
missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required training artifacts:\n' + '\n'.join(missing) + '\nFinish the fine-tuning notebook first.')

with EXPERIMENT_CONFIG_PATH.open('r', encoding='utf-8') as f:
    experiment_config = json.load(f)
with RUN_STATUS_PATH.open('r', encoding='utf-8') as f:
    run_status = json.load(f)
with SPLIT_METADATA_PATH.open('r', encoding='utf-8') as f:
    split_metadata = json.load(f)

if run_status.get('status') != 'completed':
    raise RuntimeError('Training is not marked completed. Do not evaluate the final test set yet.')

BASE_MODEL_ID = experiment_config['base_model_id']
DATASET_JSONL = Path(experiment_config['dataset_jsonl'])
DATASET_ROOT = DATASET_JSONL.parent
EXPECTED_TEST_RECORDS = int(experiment_config['split']['test_records'])

adapter_config_path = FINAL_ADAPTER_DIR / 'adapter_config.json'
if not adapter_config_path.exists():
    raise FileNotFoundError(f'Missing adapter_config.json: {adapter_config_path}')
with adapter_config_path.open('r', encoding='utf-8') as f:
    adapter_config = json.load(f)

adapter_base = adapter_config.get('base_model_name_or_path')
if adapter_base and adapter_base != BASE_MODEL_ID:
    raise RuntimeError(f'Base-model mismatch: experiment={BASE_MODEL_ID}, adapter={adapter_base}')

# Load frozen test JSONL exactly as exported by training.
def load_jsonl(path):
    records=[]
    with path.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line=line.strip()
            if not line:
                continue
            try:
                r=json.loads(line)
                r['_test_line']=line_no
                records.append(r)
            except Exception as e:
                raise ValueError(f'Invalid JSON at line {line_no}: {e}')
    return records

test_records = load_jsonl(TEST_JSONL)
if len(test_records) != EXPECTED_TEST_RECORDS:
    raise RuntimeError(f'Test count mismatch. Expected {EXPECTED_TEST_RECORDS}, found {len(test_records)}')

split_manifest = pd.read_csv(SPLIT_MANIFEST_PATH)
manifest_test = split_manifest[split_manifest['split'] == 'test'].copy()
jsonl_images = {r['image'] for r in test_records}
manifest_images = set(manifest_test['image'])
if jsonl_images != manifest_images:
    raise RuntimeError('test.jsonl and split_manifest.csv do not contain the same frozen test images.')

print('Base model:', BASE_MODEL_ID)
print('Best checkpoint:', run_status.get('best_model_checkpoint'))
print('Frozen test images:', len(test_records))
print('Buildings:', len({r['metadata']['building_id'] for r in test_records}))

## 5. Verify test images, cache them locally, and build the evaluation table

In [ ]:
def resolve_image_path(stored_path):
    stored_path = str(stored_path).strip()
    p = Path(stored_path)

    # Absolute path already stored
    if p.is_absolute() and p.exists():
        return p.resolve()

    # Relative path from the processed image dataset
    candidate = IMAGE_ROOT / stored_path.lstrip("/\\")

    if candidate.exists():
        return candidate.resolve()

    return None

resolved_test_paths={}
errors=[]
for r in tqdm(test_records, desc='Verifying test images'):
    p=resolve_image_path(r['image'])
    if p is None:
        errors.append({'image':r['image'],'error':'File not found'})
        continue
    try:
        with Image.open(p) as im:
            im=ImageOps.exif_transpose(im).convert('RGB')
            im.load()
        resolved_test_paths[r['image']] = str(p)
    except Exception as e:
        errors.append({'image':r['image'],'error':repr(e)})

if errors:
    pd.DataFrame(errors).to_csv(METRICS_DIR/'test_image_errors.csv', index=False)
    raise RuntimeError('Some test images are missing or unreadable. See test_image_errors.csv')

# Local cache reduces Google Drive I/O during repeated generation.
evaluation_image_paths={}
if CACHE_TEST_IMAGES_LOCALLY:
    LOCAL_IMAGE_CACHE.mkdir(parents=True, exist_ok=True)
    for r in tqdm(test_records, desc='Caching test images'):
        src=Path(resolved_test_paths[r['image']])
        key=hashlib.sha1(r['image'].encode('utf-8')).hexdigest()[:16]
        dst=LOCAL_IMAGE_CACHE/f'{key}_{src.name}'
        if not dst.exists():
            shutil.copy2(src,dst)
        evaluation_image_paths[r['image']] = str(dst)
else:
    evaluation_image_paths=dict(resolved_test_paths)

IMAGE_MARKER_PATTERN = re.compile(r'<image>\s*', flags=re.IGNORECASE)
def clean_instruction(x):
    return IMAGE_MARKER_PATTERN.sub('', str(x)).strip()

rows=[]
for idx,r in enumerate(test_records):
    m=r['metadata']; c=r['conversations']
    rows.append({
        'test_index': idx,
        'image': r['image'],
        'building_id': m['building_id'],
        'building_name_english': m['building_name_english'],
        'building_name_arabic': m['building_name_arabic'],
        'lighting': m['lighting'],
        'instruction': clean_instruction(c[0]['content']),
        'reference': str(c[1]['content']).strip(),
    })

test_df=pd.DataFrame(rows)
display(test_df.head())
display(test_df['lighting'].value_counts().rename_axis('lighting').reset_index(name='images'))

In [ ]:
test_image = test_records[0]["image"]

print("Stored path:")
print(test_image)

expected = IMAGE_ROOT / test_image

print("\nExpected absolute path:")
print(expected)

print("\nIMAGE_ROOT exists:")
print(IMAGE_ROOT.exists())

print("\nBuilding folder exists:")
print((IMAGE_ROOT / "001_xenel").exists())

print("\nImage exists:")
print(expected.exists())

## 6. Load the saved processor and the fine-tuned PEFT model

In [ ]:
processor = AutoProcessor.from_pretrained(str(PROCESSOR_DIR))
processor.tokenizer.padding_side='left'
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=COMPUTE_DTYPE,
    attn_implementation='sdpa',
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base_model, str(FINAL_ADAPTER_DIR), is_trainable=False)
model.eval()
model.config.use_cache=True

print('Processor:', PROCESSOR_DIR)
print('Compute dtype:', COMPUTE_DTYPE)
print(f'Allocated VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

## 7. Deterministic multimodal generation helpers with automatic OOM batch fallback

In [ ]:
def model_input_device(m):
    for p in m.parameters():
        if p.device.type != 'meta':
            return p.device
    return torch.device('cuda')

def build_prompt(instruction):
    messages=[{'role':'user','content':[{'type':'image'},{'type':'text','text':instruction}]}]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def open_rgb_image(stored_path):
    im=Image.open(evaluation_image_paths[stored_path])
    return ImageOps.exif_transpose(im).convert('RGB')

def generated_token_info(token_row):
    tokens=token_row.tolist(); eos=processor.tokenizer.eos_token_id; pad=processor.tokenizer.pad_token_id
    count=0; saw_eos=False
    for tok in tokens:
        if eos is not None and tok == eos:
            saw_eos=True; break
        if pad is not None and tok == pad:
            continue
        count += 1
    return count, bool(count >= MAX_NEW_TOKENS and not saw_eos)

def generate_batch_once(batch_df):
    prompts=[build_prompt(x) for x in batch_df['instruction'].tolist()]
    images=[open_rgb_image(x) for x in batch_df['image'].tolist()]
    try:
        inputs=processor(text=prompts, images=images, padding=True, return_tensors='pt')
        inputs.pop('token_type_ids', None)
        device=model_input_device(model)
        for k,v in list(inputs.items()):
            if isinstance(v,torch.Tensor):
                inputs[k]=v.to(device)
        input_len=inputs['input_ids'].shape[1]
        with torch.inference_mode():
            ids=model.generate(
                **inputs,
                **GENERATION_KWARGS,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )
        new_ids=ids[:,input_len:]
        texts=processor.batch_decode(new_ids,skip_special_tokens=True,clean_up_tokenization_spaces=False)
        info=[generated_token_info(row) for row in new_ids]
        return [x.strip() for x in texts], [x[0] for x in info], [x[1] for x in info]
    finally:
        for im in images:
            im.close()

def generate_batch_safe(batch_df):
    try:
        return generate_batch_once(batch_df)
    except torch.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch_df) <= 1:
            raise
        mid=len(batch_df)//2
        a=generate_batch_safe(batch_df.iloc[:mid])
        b=generate_batch_safe(batch_df.iloc[mid:])
        return a[0]+b[0], a[1]+b[1], a[2]+b[2]

print('Generation helpers ready.')

## 8. Resume-safe BASE and FINE-TUNED prediction generation

In [ ]:
PREDICTIONS_PATH = PREDICTIONS_DIR/'test_predictions.csv'

if PREDICTIONS_PATH.exists():
    predictions_df=pd.read_csv(PREDICTIONS_PATH)
    if predictions_df['image'].tolist() != test_df['image'].tolist():
        raise RuntimeError('Existing prediction file does not match the frozen test set. Use a new evaluation directory.')
    print('Loaded existing predictions; completed rows will be reused.')
else:
    predictions_df=test_df.copy()
    for col in ['base_prediction','base_generated_tokens','base_truncated','finetuned_prediction','finetuned_generated_tokens','finetuned_truncated']:
        predictions_df[col]=pd.NA
    predictions_df.to_csv(PREDICTIONS_PATH,index=False)


def run_prediction_pass(pred_col,tok_col,trunc_col,adapter_enabled):
    global predictions_df
    existing=predictions_df[pred_col].fillna('').astype(str).str.strip()
    remaining=predictions_df.index[existing==''].tolist()
    if not remaining:
        print(pred_col, ': already complete')
        return
    ctx=nullcontext() if adapter_enabled else model.disable_adapter()
    with ctx:
        for start in tqdm(range(0,len(remaining),GENERATION_BATCH_SIZE),desc=pred_col):
            idx=remaining[start:start+GENERATION_BATCH_SIZE]
            texts,counts,flags=generate_batch_safe(predictions_df.loc[idx])
            predictions_df.loc[idx,pred_col]=texts
            predictions_df.loc[idx,tok_col]=counts
            predictions_df.loc[idx,trunc_col]=flags
            predictions_df.to_csv(PREDICTIONS_PATH,index=False)

# Base = identical backbone with LoRA temporarily disabled.
run_prediction_pass('base_prediction','base_generated_tokens','base_truncated',adapter_enabled=False)
# Fine-tuned = LoRA active.
run_prediction_pass('finetuned_prediction','finetuned_generated_tokens','finetuned_truncated',adapter_enabled=True)

predictions_df=pd.read_csv(PREDICTIONS_PATH)
if predictions_df[['base_prediction','finetuned_prediction']].isna().any().any():
    raise RuntimeError('Prediction generation is incomplete.')

for col in ['base_truncated','finetuned_truncated']:
    n=predictions_df[col].astype(str).str.lower().eq('true').sum()
    print(col, ':', n)
    if n:
        print('WARNING: Some outputs reached MAX_NEW_TOKENS. Inspect before final paper reporting.')

print('Prediction file:', PREDICTIONS_PATH)

## 9. Release Qwen3-VL from GPU before BERTScore

In [ ]:
del model
del base_model
gc.collect()
torch.cuda.empty_cache()
print(f'Allocated VRAM after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

## 10. Compute BLEU-4, ROUGE-L, METEOR, BERTScore, and CIDEr

In [ ]:
TOKEN_PATTERN=re.compile(r"\b[\w’'-]+\b",flags=re.UNICODE)
smoothing=SmoothingFunction().method4
rouge=rouge_scorer.RougeScorer(['rougeL'],use_stemmer=True)

def tokens(text): return [t.lower() for t in TOKEN_PATTERN.findall(str(text))]
def sent_bleu4(ref,pred):
    p=tokens(pred)
    return 0.0 if not p else float(sentence_bleu([tokens(ref)],p,weights=(.25,.25,.25,.25),smoothing_function=smoothing))
def rouge_l(ref,pred): return float(rouge.score(str(ref),str(pred))['rougeL'].fmeasure)
def meteor_v(ref,pred):
    p=tokens(pred)
    return 0.0 if not p else float(meteor_score([tokens(ref)],p))

def corpus_bleu4(refs,preds):
    return float(corpus_bleu([[tokens(r)] for r in refs],[tokens(p) for p in preds],weights=(.25,.25,.25,.25),smoothing_function=smoothing))

refs=predictions_df['reference'].astype(str).tolist()
for prefix in ['base','finetuned']:
    preds=predictions_df[f'{prefix}_prediction'].astype(str).tolist()
    predictions_df[f'{prefix}_sentence_bleu4']=[sent_bleu4(r,p) for r,p in zip(refs,preds)]
    predictions_df[f'{prefix}_rougeL']=[rouge_l(r,p) for r,p in zip(refs,preds)]
    predictions_df[f'{prefix}_meteor']=[meteor_v(r,p) for r,p in zip(refs,preds)]

# BERTScore on GPU after VLM cleanup.
def bert_f1(preds):
    _,_,f1=bertscore_score(
        cands=preds, refs=refs, model_type=BERTSCORE_MODEL, lang='en',
        batch_size=BERTSCORE_BATCH_SIZE,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        rescale_with_baseline=BERTSCORE_RESCALE_WITH_BASELINE, verbose=True,
    )
    return f1.detach().cpu().numpy().astype(float)

predictions_df['base_bertscore_f1']=bert_f1(predictions_df['base_prediction'].astype(str).tolist())
gc.collect(); torch.cuda.empty_cache()
predictions_df['finetuned_bertscore_f1']=bert_f1(predictions_df['finetuned_prediction'].astype(str).tolist())
gc.collect(); torch.cuda.empty_cache()

# CIDEr using the COCO caption evaluation implementation.
def compute_cider(refs,preds):
    try:
        from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
        from pycocoevalcap.cider.cider import Cider
        gts_raw={str(i):[{'caption':r}] for i,r in enumerate(refs)}
        res_raw={str(i):[{'caption':p}] for i,p in enumerate(preds)}
        tok=PTBTokenizer(); gts=tok.tokenize(gts_raw); res=tok.tokenize(res_raw)
        corpus,scores=Cider().compute_score(gts,res)
        return float(corpus),np.asarray(scores,dtype=float)
    except Exception as e:
        warnings.warn(f'CIDEr failed: {e}')
        return float('nan'),np.full(len(refs),np.nan)

base_cider_corpus,base_cider=compute_cider(refs,predictions_df['base_prediction'].astype(str).tolist())
ft_cider_corpus,ft_cider=compute_cider(refs,predictions_df['finetuned_prediction'].astype(str).tolist())
predictions_df['base_cider']=base_cider
predictions_df['finetuned_cider']=ft_cider

print('Base corpus BLEU-4:', corpus_bleu4(refs,predictions_df['base_prediction']))
print('Fine-tuned corpus BLEU-4:', corpus_bleu4(refs,predictions_df['finetuned_prediction']))
print('Base single-reference CIDEr:', base_cider_corpus)
print('Fine-tuned single-reference CIDEr:', ft_cider_corpus)

## 11. Building-identification metric and output-length analysis

In [ ]:
def normalize_name(text):
    text=str(text).lower().replace('’',"'")
    text=re.sub(r'[-_/]',' ',text)
    text=re.sub(r"[^\w\u0600-\u06FF\s']",' ',text,flags=re.UNICODE)
    return re.sub(r'\s+',' ',text).strip()

def english_core(name):
    x=normalize_name(name)
    for prefix in ['bayt ','house ','ribat ','waqf ']:
        if x.startswith(prefix):
            return x[len(prefix):].strip()
    return x

building_table=predictions_df[['building_id','building_name_english','building_name_arabic']].drop_duplicates()
core_to_ids=defaultdict(set)
for _,r in building_table.iterrows(): core_to_ids[english_core(r['building_name_english'])].add(r['building_id'])

building_aliases={}
for _,r in building_table.iterrows():
    bid=r['building_id']; en=normalize_name(r['building_name_english']); ar=normalize_name(r['building_name_arabic']); core=english_core(r['building_name_english'])
    aliases={en}
    if ar: aliases.add(ar)
    if en.startswith('bayt ') and core:
        aliases.update({normalize_name(f'house {core}'),normalize_name(f'{core} house')})
    if core and len(core)>=4 and len(core_to_ids[core])==1: aliases.add(core)
    building_aliases[bid]=sorted({a for a in aliases if len(a)>=3},key=len,reverse=True)

def contains_alias(text,alias):
    return bool(re.search(r'(?<!\w)'+re.escape(alias)+r'(?!\w)',text,flags=re.UNICODE))

def identify_building(text):
    t=normalize_name(text); matches=[]
    for bid,aliases in building_aliases.items():
        for a in aliases:
            if contains_alias(t,a):
                matches.append((len(a),bid,a)); break
    if not matches: return None,None,'no_name_detected'
    matches.sort(reverse=True); best_len=matches[0][0]; best=[m for m in matches if m[0]==best_len]
    if len({m[1] for m in best})>1: return None,None,'ambiguous_multiple_buildings'
    _,bid,a=best[0]; return bid,a,'identified'

for prefix in ['base','finetuned']:
    found=[identify_building(x) for x in predictions_df[f'{prefix}_prediction']]
    predictions_df[f'{prefix}_predicted_building_id']=[x[0] for x in found]
    predictions_df[f'{prefix}_matched_building_alias']=[x[1] for x in found]
    predictions_df[f'{prefix}_building_identification_status']=[x[2] for x in found]
    predictions_df[f'{prefix}_building_correct']=predictions_df[f'{prefix}_predicted_building_id']==predictions_df['building_id']

def word_count(text): return len(tokens(text))
predictions_df['reference_words']=predictions_df['reference'].map(word_count)
predictions_df['base_words']=predictions_df['base_prediction'].map(word_count)
predictions_df['finetuned_words']=predictions_df['finetuned_prediction'].map(word_count)

display(pd.DataFrame({
    'model':['Base','Fine-tuned'],
    'building_accuracy':[predictions_df['base_building_correct'].mean(),predictions_df['finetuned_building_correct'].mean()],
    'mean_words':[predictions_df['base_words'].mean(),predictions_df['finetuned_words'].mean()],
}))

## 12. Save per-image metrics and compute overall, lighting, and per-building results

In [ ]:
PER_IMAGE_METRICS_PATH=METRICS_DIR/'predictions_and_per_image_metrics.csv'
predictions_df.to_csv(PER_IMAGE_METRICS_PATH,index=False)

def macro_building_accuracy(df,col): return float(df.groupby('building_id')[col].mean().mean())

base_corpus_bleu4=corpus_bleu4(refs,predictions_df['base_prediction'].astype(str).tolist())
ft_corpus_bleu4=corpus_bleu4(refs,predictions_df['finetuned_prediction'].astype(str).tolist())

overall_metrics=pd.DataFrame([
    {
        'model':'Base','corpus_bleu4':base_corpus_bleu4,
        'sentence_bleu4_mean':predictions_df['base_sentence_bleu4'].mean(),
        'rougeL_f1':predictions_df['base_rougeL'].mean(),'meteor':predictions_df['base_meteor'].mean(),
        'bertscore_f1':predictions_df['base_bertscore_f1'].mean(),'cider_single_reference':base_cider_corpus,
        'building_accuracy':predictions_df['base_building_correct'].mean(),
        'macro_building_accuracy':macro_building_accuracy(predictions_df,'base_building_correct'),
        'mean_output_words':predictions_df['base_words'].mean(),
    },
    {
        'model':'Fine-tuned','corpus_bleu4':ft_corpus_bleu4,
        'sentence_bleu4_mean':predictions_df['finetuned_sentence_bleu4'].mean(),
        'rougeL_f1':predictions_df['finetuned_rougeL'].mean(),'meteor':predictions_df['finetuned_meteor'].mean(),
        'bertscore_f1':predictions_df['finetuned_bertscore_f1'].mean(),'cider_single_reference':ft_cider_corpus,
        'building_accuracy':predictions_df['finetuned_building_correct'].mean(),
        'macro_building_accuracy':macro_building_accuracy(predictions_df,'finetuned_building_correct'),
        'mean_output_words':predictions_df['finetuned_words'].mean(),
    }
])
overall_metrics.to_csv(METRICS_DIR/'overall_metrics.csv',index=False)

lighting_rows=[]
for lighting,g in predictions_df.groupby('lighting',sort=True):
    for label,prefix in [('Base','base'),('Fine-tuned','finetuned')]:
        lighting_rows.append({
            'lighting':lighting,'model':label,'images':len(g),
            'sentence_bleu4_mean':g[f'{prefix}_sentence_bleu4'].mean(),
            'rougeL_f1':g[f'{prefix}_rougeL'].mean(),'meteor':g[f'{prefix}_meteor'].mean(),
            'bertscore_f1':g[f'{prefix}_bertscore_f1'].mean(),
            'cider_mean_single_reference':g[f'{prefix}_cider'].mean(),
            'building_accuracy':g[f'{prefix}_building_correct'].mean(),
        })
metrics_by_lighting=pd.DataFrame(lighting_rows)
metrics_by_lighting.to_csv(METRICS_DIR/'metrics_by_lighting.csv',index=False)

pb=[]
for (bid,bname),g in predictions_df.groupby(['building_id','building_name_english'],sort=True):
    row={'building_id':bid,'building_name_english':bname,'test_images':len(g),
         'day_images':int(g['lighting'].astype(str).str.lower().eq('day').sum()),
         'night_images':int(g['lighting'].astype(str).str.lower().eq('night').sum())}
    for prefix in ['base','finetuned']:
        row.update({
            f'{prefix}_sentence_bleu4':g[f'{prefix}_sentence_bleu4'].mean(),
            f'{prefix}_rougeL':g[f'{prefix}_rougeL'].mean(),f'{prefix}_meteor':g[f'{prefix}_meteor'].mean(),
            f'{prefix}_bertscore_f1':g[f'{prefix}_bertscore_f1'].mean(),
            f'{prefix}_building_accuracy':g[f'{prefix}_building_correct'].mean(),
        })
    row['building_accuracy_change']=row['finetuned_building_accuracy']-row['base_building_accuracy']
    row['bertscore_change']=row['finetuned_bertscore_f1']-row['base_bertscore_f1']
    pb.append(row)
per_building_metrics=pd.DataFrame(pb).sort_values(['finetuned_building_accuracy','finetuned_bertscore_f1'],ascending=False)
per_building_metrics.to_csv(METRICS_DIR/'per_building_metrics.csv',index=False)

display(overall_metrics.round(4))
display(metrics_by_lighting.round(4))

## 13. Paired bootstrap confidence intervals and significance tests

In [ ]:
def paired_bootstrap(base,ft,iterations=BOOTSTRAP_ITERATIONS,confidence=BOOTSTRAP_CONFIDENCE,seed=SEED):
    base=np.asarray(base,dtype=float); ft=np.asarray(ft,dtype=float)
    valid=np.isfinite(base)&np.isfinite(ft); base=base[valid]; ft=ft[valid]; n=len(base)
    if not n: return {'n':0,'base_mean':np.nan,'finetuned_mean':np.nan,'delta':np.nan,'ci_low':np.nan,'ci_high':np.nan,'probability_delta_gt_zero':np.nan}
    rng=np.random.default_rng(seed); deltas=np.empty(iterations)
    for i in range(iterations):
        idx=rng.integers(0,n,size=n); deltas[i]=ft[idx].mean()-base[idx].mean()
    alpha=1-confidence
    return {'n':n,'base_mean':base.mean(),'finetuned_mean':ft.mean(),'delta':ft.mean()-base.mean(),
            'ci_low':np.quantile(deltas,alpha/2),'ci_high':np.quantile(deltas,1-alpha/2),
            'probability_delta_gt_zero':(deltas>0).mean()}

pairs={
    'Sentence BLEU-4':('base_sentence_bleu4','finetuned_sentence_bleu4'),
    'ROUGE-L F1':('base_rougeL','finetuned_rougeL'),
    'METEOR':('base_meteor','finetuned_meteor'),
    'BERTScore F1':('base_bertscore_f1','finetuned_bertscore_f1'),
    'CIDEr (single-reference)':('base_cider','finetuned_cider'),
    'Building identification':('base_building_correct','finetuned_building_correct'),
}
boot=[]
for name,(b,f) in pairs.items():
    r=paired_bootstrap(predictions_df[b].astype(float),predictions_df[f].astype(float)); r['metric']=name; boot.append(r)
bootstrap_results=pd.DataFrame(boot)[['metric','n','base_mean','finetuned_mean','delta','ci_low','ci_high','probability_delta_gt_zero']]
bootstrap_results.to_csv(METRICS_DIR/'paired_bootstrap_95ci.csv',index=False)

sig=[]
for name,(b,f) in {k:v for k,v in pairs.items() if k in ['Sentence BLEU-4','ROUGE-L F1','METEOR','BERTScore F1']}.items():
    bv=predictions_df[b].astype(float).to_numpy(); fv=predictions_df[f].astype(float).to_numpy(); diff=fv-bv
    if np.allclose(diff,0): stat,p=0.0,1.0
    else:
        t=wilcoxon(fv,bv,alternative='two-sided',zero_method='wilcox'); stat,p=float(t.statistic),float(t.pvalue)
    sig.append({'metric':name,'test':'Wilcoxon signed-rank','statistic':stat,'p_value':p})

bc=predictions_df['base_building_correct'].astype(bool); fc=predictions_df['finetuned_building_correct'].astype(bool)
wrong_to_correct=int(((~bc)&fc).sum()); correct_to_wrong=int((bc&(~fc)).sum()); discordant=wrong_to_correct+correct_to_wrong
mcnemar_p=1.0 if discordant==0 else float(binomtest(k=min(wrong_to_correct,correct_to_wrong),n=discordant,p=.5).pvalue)
sig.append({'metric':'Building identification','test':'Exact McNemar','statistic':wrong_to_correct-correct_to_wrong,'p_value':mcnemar_p})
significance_results=pd.DataFrame(sig)
significance_results.to_csv(METRICS_DIR/'paired_significance_tests.csv',index=False)

predictions_df['building_transition']=[
    'Both correct' if b and f else
    'Base wrong → Fine-tuned correct' if (not b) and f else
    'Base correct → Fine-tuned wrong' if b and (not f) else
    'Both wrong'
    for b,f in zip(bc,fc)
]
transition_summary=predictions_df['building_transition'].value_counts().rename_axis('outcome').reset_index(name='images')
transition_summary.to_csv(METRICS_DIR/'building_identification_transitions.csv',index=False)

display(bootstrap_results.round(5))
display(significance_results.round(6))
print('Base wrong → Fine-tuned correct:', wrong_to_correct)
print('Base correct → Fine-tuned wrong:', correct_to_wrong)

## 14. Paper-ready quantitative figures

In [ ]:
# 1) Main caption metrics (0-1 family only; CIDEr shown in tables because scale differs).
metric_plot=pd.DataFrame({
    'Metric':['Corpus BLEU-4','ROUGE-L','METEOR','BERTScore F1'],
    'Base':[overall_metrics.loc[0,'corpus_bleu4'],overall_metrics.loc[0,'rougeL_f1'],overall_metrics.loc[0,'meteor'],overall_metrics.loc[0,'bertscore_f1']],
    'Fine-tuned':[overall_metrics.loc[1,'corpus_bleu4'],overall_metrics.loc[1,'rougeL_f1'],overall_metrics.loc[1,'meteor'],overall_metrics.loc[1,'bertscore_f1']],
})
ax=metric_plot.set_index('Metric').plot(kind='bar',figsize=(10,6))
ax.set_ylabel('Score'); ax.set_xlabel(''); ax.set_title('Base vs Fine-Tuned Caption Evaluation'); ax.tick_params(axis='x',rotation=0); ax.grid(axis='y',alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'figure_overall_caption_metrics.png',dpi=300,bbox_inches='tight'); plt.show()

# 2) Overall and macro building accuracy.
acc=overall_metrics[['model','building_accuracy','macro_building_accuracy']].rename(columns={'building_accuracy':'Overall accuracy','macro_building_accuracy':'Macro building accuracy'}).set_index('model')
ax=acc.plot(kind='bar',figsize=(8,5)); ax.set_ylabel('Accuracy'); ax.set_xlabel(''); ax.set_ylim(0,1); ax.set_title('Building Identification Accuracy'); ax.tick_params(axis='x',rotation=0); ax.grid(axis='y',alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'figure_building_identification_accuracy.png',dpi=300,bbox_inches='tight'); plt.show()

# 3) Day/night building accuracy.
light=metrics_by_lighting[['lighting','model','building_accuracy']].pivot(index='lighting',columns='model',values='building_accuracy')
ax=light.plot(kind='bar',figsize=(8,5)); ax.set_ylabel('Building identification accuracy'); ax.set_xlabel('Lighting condition'); ax.set_ylim(0,1); ax.set_title('Building Identification by Lighting Condition'); ax.tick_params(axis='x',rotation=0); ax.grid(axis='y',alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'figure_day_night_building_accuracy.png',dpi=300,bbox_inches='tight'); plt.show()

# 4) Per-building accuracy.
pbplot=per_building_metrics[['building_name_english','base_building_accuracy','finetuned_building_accuracy']].sort_values(['finetuned_building_accuracy','base_building_accuracy']).set_index('building_name_english').rename(columns={'base_building_accuracy':'Base','finetuned_building_accuracy':'Fine-tuned'})
ax=pbplot.plot(kind='barh',figsize=(11,14)); ax.set_xlabel('Building identification accuracy'); ax.set_ylabel(''); ax.set_xlim(0,1); ax.set_title('Per-Building Identification Accuracy'); ax.grid(axis='x',alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'figure_per_building_identification_accuracy.png',dpi=300,bbox_inches='tight'); plt.show()

# 5) Per-image BERTScore delta.
delta=predictions_df['finetuned_bertscore_f1']-predictions_df['base_bertscore_f1']
plt.figure(figsize=(9,5)); plt.hist(delta,bins=25,edgecolor='black'); plt.axvline(0,linestyle='--',linewidth=1.5)
plt.xlabel('Fine-tuned BERTScore F1 − Base BERTScore F1'); plt.ylabel('Number of test images'); plt.title('Distribution of Per-Image Semantic Improvement'); plt.grid(axis='y',alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'figure_bertscore_improvement_distribution.png',dpi=300,bbox_inches='tight'); plt.show()

# 6) Building-ID transitions.
order=['Both correct','Base wrong → Fine-tuned correct','Base correct → Fine-tuned wrong','Both wrong']
tp=transition_summary.set_index('outcome').reindex(order,fill_value=0)
ax=tp.plot(kind='bar',y='images',legend=False,figsize=(10,5)); ax.set_ylabel('Number of test images'); ax.set_xlabel(''); ax.set_title('Building Identification Outcome Changes After Fine-Tuning'); ax.tick_params(axis='x',rotation=20); ax.grid(axis='y',alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'figure_building_identification_transitions.png',dpi=300,bbox_inches='tight'); plt.show()

# 7) Output lengths.
plt.figure(figsize=(8,5)); plt.boxplot([predictions_df['reference_words'],predictions_df['base_words'],predictions_df['finetuned_words']],tick_labels=['Reference','Base','Fine-tuned'],showfliers=False)
plt.ylabel('Description length (words)'); plt.title('Reference and Generated Description Length'); plt.grid(axis='y',alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'figure_output_length_comparison.png',dpi=300,bbox_inches='tight'); plt.show()

## 15. Objective qualitative examples and paper-ready visual comparison figures

In [ ]:
q=predictions_df.copy(); q['bertscore_delta']=q['finetuned_bertscore_f1']-q['base_bertscore_f1']
selected=[]
def add(i):
    if i is not None and i not in selected: selected.append(i)
add(q['bertscore_delta'].idxmax()); add(q['bertscore_delta'].idxmin())
med=q['bertscore_delta'].median(); add((q['bertscore_delta']-med).abs().idxmin())
identity_gain=q[(~q['base_building_correct'].astype(bool)) & q['finetuned_building_correct'].astype(bool)]
if not identity_gain.empty: add(identity_gain['bertscore_delta'].idxmax())
for lighting in ['night','day']:
    g=q[q['lighting'].astype(str).str.lower().eq(lighting)]
    if not g.empty: add(g['bertscore_delta'].idxmax())
selected_qualitative=q.loc[selected].copy()
selected_qualitative.to_csv(QUALITATIVE_DIR/'selected_qualitative_examples.csv',index=False)
display(selected_qualitative[['image','building_name_english','lighting','bertscore_delta','base_building_correct','finetuned_building_correct']])

def wrapped(x,width=95): return fill(str(x),width=width)
for n,(_,row) in enumerate(selected_qualitative.iterrows(),start=1):
    im=Image.open(evaluation_image_paths[row['image']]); im=ImageOps.exif_transpose(im).convert('RGB')
    try:
        fig=plt.figure(figsize=(12,10)); grid=fig.add_gridspec(4,1,height_ratios=[3,1.5,1.5,1.5])
        ax=fig.add_subplot(grid[0]); ax.imshow(im); ax.axis('off'); ax.set_title(f"{row['building_name_english']} | {row['lighting']} | {row['image']}",fontsize=12)
        for gi,(title,text) in enumerate([('Reference',row['reference']),('Base model',row['base_prediction']),('Fine-tuned model',row['finetuned_prediction'])],start=1):
            ta=fig.add_subplot(grid[gi]); ta.axis('off'); ta.text(0,1,f'{title}\n{wrapped(text)}',va='top',ha='left',fontsize=10,transform=ta.transAxes)
        plt.tight_layout(); plt.savefig(QUALITATIVE_DIR/f'qualitative_example_{n:02d}.png',dpi=300,bbox_inches='tight'); plt.show()
    finally:
        im.close()

## 16. Create a blinded human-evaluation sheet

In [ ]:
def balanced_sample(df,target,seed=SEED):
    rng=np.random.default_rng(seed); selected=[]
    for _,g in df.groupby('building_id',sort=True): selected.append(int(rng.choice(g.index.to_numpy(),1)[0]))
    selected=list(dict.fromkeys(selected)); remain=min(target,len(df))-len(selected)
    if remain>0:
        pool=df.drop(index=selected).index.to_numpy(); rng.shuffle(pool); selected.extend(pool[:remain].tolist())
    return df.loc[selected].copy()

sample=balanced_sample(predictions_df,HUMAN_EVAL_SAMPLE_SIZE)
rng=np.random.default_rng(SEED); rater=[]; mapping=[]
for n,(_,r) in enumerate(sample.iterrows(),start=1):
    base_is_a=bool(rng.integers(0,2)); item=f'HE_{n:03d}'
    if base_is_a:
        a,b=r['base_prediction'],r['finetuned_prediction']; asrc,bsrc='Base','Fine-tuned'
    else:
        a,b=r['finetuned_prediction'],r['base_prediction']; asrc,bsrc='Fine-tuned','Base'
    rater.append({
        'item_id':item,'image':r['image'],'building_id':r['building_id'],'building_name_english':r['building_name_english'],'lighting':r['lighting'],'reference':r['reference'],
        'model_a_text':a,'model_b_text':b,
        'model_a_building_identity_0_1':'','model_a_architectural_correctness_1_5':'','model_a_visual_grounding_1_5':'','model_a_historical_factuality_1_5':'','model_a_fluency_1_5':'','model_a_hallucination_severity_1_5':'',
        'model_b_building_identity_0_1':'','model_b_architectural_correctness_1_5':'','model_b_visual_grounding_1_5':'','model_b_historical_factuality_1_5':'','model_b_fluency_1_5':'','model_b_hallucination_severity_1_5':'','rater_notes':''
    })
    mapping.append({'item_id':item,'model_a_source':asrc,'model_b_source':bsrc})

pd.DataFrame(rater).to_csv(HUMAN_EVAL_DIR/'human_evaluation_blinded.csv',index=False)
pd.DataFrame(mapping).to_csv(HUMAN_EVAL_DIR/'human_evaluation_secret_mapping.csv',index=False)
print('Give raters:', HUMAN_EVAL_DIR/'human_evaluation_blinded.csv')
print('Keep private:', HUMAN_EVAL_DIR/'human_evaluation_secret_mapping.csv')

## 17. Main paper table, numerical results summary, and evaluation manifest

In [ ]:
paper_table=overall_metrics[['model','corpus_bleu4','rougeL_f1','meteor','bertscore_f1','cider_single_reference','building_accuracy','macro_building_accuracy']].copy()
base_row=paper_table[paper_table['model']=='Base'].iloc[0]
ft_row=paper_table[paper_table['model']=='Fine-tuned'].iloc[0]
improve={'model':'Fine-tuned − Base'}
for col in paper_table.columns:
    if col!='model': improve[col]=ft_row[col]-base_row[col]
paper_table=pd.concat([paper_table,pd.DataFrame([improve])],ignore_index=True)
paper_table.to_csv(METRICS_DIR/'paper_main_results_table.csv',index=False)

bert_ci=bootstrap_results[bootstrap_results['metric']=='BERTScore F1'].iloc[0]
bld_ci=bootstrap_results[bootstrap_results['metric']=='Building identification'].iloc[0]

summary = (
    f"Held-out test evaluation included {len(predictions_df)} images from {predictions_df['building_id'].nunique()} Historic Jeddah buildings. "
    f"The base Qwen3-VL-2B-Instruct model achieved corpus BLEU-4={base_row['corpus_bleu4']:.4f}, ROUGE-L F1={base_row['rougeL_f1']:.4f}, "
    f"METEOR={base_row['meteor']:.4f}, BERTScore F1={base_row['bertscore_f1']:.4f}, and building-identification accuracy={base_row['building_accuracy']:.2%}. "
    f"After Al-Balad fine-tuning, the corresponding scores were BLEU-4={ft_row['corpus_bleu4']:.4f}, ROUGE-L F1={ft_row['rougeL_f1']:.4f}, "
    f"METEOR={ft_row['meteor']:.4f}, BERTScore F1={ft_row['bertscore_f1']:.4f}, and building-identification accuracy={ft_row['building_accuracy']:.2%}. "
    f"The paired mean BERTScore change was {bert_ci['delta']:+.4f} with a {BOOTSTRAP_CONFIDENCE:.0%} bootstrap CI [{bert_ci['ci_low']:+.4f}, {bert_ci['ci_high']:+.4f}]. "
    f"The paired building-identification change was {bld_ci['delta']:+.2%} with a {BOOTSTRAP_CONFIDENCE:.0%} bootstrap CI [{bld_ci['ci_low']:+.2%}, {bld_ci['ci_high']:+.2%}].\n\n"
    "Interpretation note: the test references are synthetic Qwen3-VL descriptions generated with building metadata and survey context. "
    "Caption-similarity metrics therefore quantify agreement with the synthetic references and are not independent evidence of historical factual correctness."
)
RESULTS_SUMMARY_PATH=EVALUATION_ROOT/'research_results_summary.txt'
RESULTS_SUMMARY_PATH.write_text(summary,encoding='utf-8')

config={
    'experiment_name':EXPERIMENT_NAME,'base_model_id':BASE_MODEL_ID,'adapter_dir':str(FINAL_ADAPTER_DIR),'processor_dir':str(PROCESSOR_DIR),
    'test_jsonl':str(TEST_JSONL),'test_records':len(predictions_df),'unique_buildings':int(predictions_df['building_id'].nunique()),
    'generation':{'batch_size':GENERATION_BATCH_SIZE,**GENERATION_KWARGS},
    'quantization':{'load_in_4bit':True,'quant_type':'nf4','double_quantization':True,'compute_dtype':str(COMPUTE_DTYPE)},
    'bertscore':{'model_type':BERTSCORE_MODEL,'batch_size':BERTSCORE_BATCH_SIZE,'rescale_with_baseline':BERTSCORE_RESCALE_WITH_BASELINE},
    'statistics':{'bootstrap_iterations':BOOTSTRAP_ITERATIONS,'bootstrap_confidence':BOOTSTRAP_CONFIDENCE,'seed':SEED},
    'reference_caption_source':'Synthetic Qwen3-VL descriptions generated with building metadata and survey context.',
    'single_reference_cider':True,
    'training_clean_dataset_fingerprint':split_metadata.get('clean_dataset_fingerprint'),
}
(EVALUATION_ROOT/'evaluation_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

manifest={
    'predictions':str(PREDICTIONS_PATH),'per_image_metrics':str(PER_IMAGE_METRICS_PATH),
    'overall_metrics':str(METRICS_DIR/'overall_metrics.csv'),'lighting_metrics':str(METRICS_DIR/'metrics_by_lighting.csv'),
    'per_building_metrics':str(METRICS_DIR/'per_building_metrics.csv'),'bootstrap_results':str(METRICS_DIR/'paired_bootstrap_95ci.csv'),
    'significance_tests':str(METRICS_DIR/'paired_significance_tests.csv'),'paper_table':str(METRICS_DIR/'paper_main_results_table.csv'),
    'figures':str(FIGURES_DIR),'qualitative_examples':str(QUALITATIVE_DIR),'human_evaluation':str(HUMAN_EVAL_DIR),'results_summary':str(RESULTS_SUMMARY_PATH),
}
(EVALUATION_ROOT/'evaluation_output_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')

# Save the final per-image table again because later sections added building transitions.
predictions_df.to_csv(PER_IMAGE_METRICS_PATH,index=False)

display(paper_table.round(4))
print('\n'+summary)
print('\nEvaluation folder:', EVALUATION_ROOT)

## 18. Output structure

After completion:

```text
evaluation_v1/
├── predictions/
│   └── test_predictions.csv
├── metrics/
│   ├── predictions_and_per_image_metrics.csv
│   ├── overall_metrics.csv
│   ├── metrics_by_lighting.csv
│   ├── per_building_metrics.csv
│   ├── paired_bootstrap_95ci.csv
│   ├── paired_significance_tests.csv
│   ├── building_identification_transitions.csv
│   └── paper_main_results_table.csv
├── figures/
│   ├── figure_overall_caption_metrics.png
│   ├── figure_building_identification_accuracy.png
│   ├── figure_day_night_building_accuracy.png
│   ├── figure_per_building_identification_accuracy.png
│   ├── figure_bertscore_improvement_distribution.png
│   ├── figure_building_identification_transitions.png
│   └── figure_output_length_comparison.png
├── qualitative_examples/
│   ├── selected_qualitative_examples.csv
│   └── qualitative_example_*.png
├── human_evaluation/
│   ├── human_evaluation_blinded.csv
│   └── human_evaluation_secret_mapping.csv
├── research_results_summary.txt
├── evaluation_config.json
└── evaluation_output_manifest.json
```

The following notebook, `07_external_image_testing_albalad_v1`, should be treated as an additional generalization experiment and **not** as a replacement for this frozen held-out test evaluation.